Əvvəl notebooku və SQL bağlantısını hazırlayırıq.

In [124]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from db import get_engine

engine = get_engine()

with engine.connect() as conn:
    db_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Python və SQL Server hazırdır:", db_name)

Python və SQL Server hazırdır: BankAnalyticsDB


Kredit məlumatlarını oxuyaq

In [78]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

In [79]:
silver_loans_df.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2026-01-01,1,Yeni il,0,0,0,2026-01-01,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,2026-01-03,0,NaN,1,0,0,2026-01-03,3.6,0.0,10.7


In [80]:
silver_loans_df.shape

(4820, 32)

In [81]:
silver_loans_df.duplicated().sum()

np.int64(15)

In [82]:
silver_loans_df = silver_loans_df.drop_duplicates()

In [83]:
silver_loans_df.duplicated().sum()

np.int64(0)

In [84]:
silver_loans_df.isna().sum()

loan_id                   0
customer_id               0
application_date          0
product_type              0
amount                    0
term_months               0
interest_rate            50
status                    0
rejection_reason       3070
defaulted              1826
age                       5
gender                    5
city                      5
branch                    5
tenure_years              5
credit_score              5
balance                   5
salary                    5
products_count            5
has_card                  5
is_active                 5
churned                   5
merge_date                0
is_holiday                0
holiday_name           4768
is_weekend                0
is_workday                0
is_pre_holiday            0
time                      5
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64

In [85]:
silver_loans_df.groupby("status")["defaulted"].apply(lambda x: x.isna().sum())

status
Rədd edildi    1777
Təsdiqləndi       0
rədd edildi      49
təsdiqləndi       0
Name: defaulted, dtype: int64

defaulted kredit gecikmesini gosterir. ve 1826 bos deyerin hamisi redd edilen kreditlere aiddir. bu normaldir. 

Status yazılışlarını eyniləşdirək

In [86]:
silver_loans_df["status"] = silver_loans_df["status"].str.lower()

In [87]:
silver_loans_df["status"].value_counts()

status
təsdiqləndi    2979
rədd edildi    1826
Name: count, dtype: int64

interest rate de bos olan 50 deyere baxaq


In [88]:
silver_loans_df[silver_loans_df["interest_rate"].isna()]["status"].value_counts(dropna=False)

status
təsdiqləndi    34
rədd edildi    16
Name: count, dtype: int64

34-ü təsdiqlənib — bunlarda faiz dərəcəsi olmalıdır.
16-sı rədd edilib — bunlarda faizin boş qalması normal ola bilər.

kredit novlerinin medianlarini tapaq ve bu deyerlere uygun bosluqlari dolduraq

In [89]:
silver_loans_df.groupby("product_type")["interest_rate"].median()

product_type
Avtomobil krediti    11.00
Biznes krediti       12.13
İpoteka               8.09
İstehlak krediti     14.06
Name: interest_rate, dtype: float64

In [90]:
mask = silver_loans_df["status"] == "təsdiqləndi"
medians = silver_loans_df.groupby("product_type")["interest_rate"].transform("median")

silver_loans_df.loc[mask, "interest_rate"] = silver_loans_df.loc[mask, "interest_rate"].fillna(medians)

yerde qalan bos deyerlerin sayina baxaq

In [91]:
silver_loans_df["interest_rate"].isna().sum()

np.int64(16)

novbeti sutunlarin bos deyerlerine baxaq

In [92]:
silver_loans_df[silver_loans_df["age"].isna()][
    ["loan_id", "customer_id", "age", "gender", "city", "salary"]
]

,loan_id,customer_id,age,gender,city,salary
1402,L001398,C09999,NaN,NaN,NaN,NaN
1754,L001749,C09999,NaN,NaN,NaN,NaN
2672,L002665,C09999,NaN,NaN,NaN,NaN
3698,L003687,C09999,NaN,NaN,NaN,NaN
3795,L003784,C09999,NaN,NaN,NaN,NaN


In [93]:
pd.read_sql(
    "SELECT * FROM bronze.customers WHERE customer_id = 'C09999'",
    engine
)

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned


yalniz bir musterinin adi customer cedvelinde yoxdu

baxaq ki bu musterinin tesdiqlenmis krediti varmi?

In [94]:
silver_loans_df[silver_loans_df["customer_id"] == "C09999"]["status"].value_counts()

status
təsdiqləndi    4
rədd edildi    1
Name: count, dtype: int64

bu 5 krediti silməyək. Çünki kreditlərin öz məlumatları mövcuddur, sadəcə müştəri məlumatları çatışmır. Silmək kredit sayını və ümumi məbləği səhv göstərə bilər.

In [95]:
silver_loans_df["customer_missing"] = silver_loans_df["age"].isna()

silver_loans_df["customer_missing"].value_counts()

customer_missing
False    4800
True        5
Name: count, dtype: int64

In [96]:
silver_loans_df.isna().sum()[lambda x: x > 0]

interest_rate            16
rejection_reason       3070
defaulted              1826
age                       5
gender                    5
city                      5
branch                    5
tenure_years              5
credit_score              5
balance                   5
salary                    5
products_count            5
has_card                  5
is_active                 5
churned                   5
holiday_name           4768
time                      5
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64

In [97]:
silver_loans_df.groupby("is_holiday")["holiday_name"].apply(lambda x: x.isna().sum())

is_holiday
0    4768
1       0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyer yoxdur

In [98]:
silver_loans_df.shape


(4805, 33)

In [99]:
silver_loans_df.dtypes

loan_id                    str
customer_id                str
application_date           str
product_type               str
amount                 float64
term_months              int64
interest_rate          float64
status                     str
rejection_reason           str
defaulted              float64
age                    float64
gender                     str
city                       str
branch                     str
tenure_years           float64
credit_score           float64
balance                float64
salary                 float64
products_count         float64
has_card               float64
is_active              float64
churned                float64
merge_date              object
is_holiday               int64
holiday_name               str
is_weekend               int64
is_workday               int64
is_pre_holiday           int64
time                       str
temperature_2m_mean    float64
precipitation_sum      float64
wind_speed_10m_max     float64
customer

In [100]:
silver_loans_df["application_date"] = pd.to_datetime(
    silver_loans_df["application_date"],
    errors="coerce"
)

silver_loans_df["merge_date"] = pd.to_datetime(
    silver_loans_df["merge_date"],
    errors="coerce"
)

In [101]:
sutunlar = [
    "age", "credit_score", "products_count",
    "has_card", "is_active", "churned"
]

silver_loans_df[sutunlar] = silver_loans_df[sutunlar].astype("Int64")

In [102]:
silver_loans_df["defaulted"] = silver_loans_df["defaulted"].astype("Int64")
silver_loans_df["tenure_years"] = silver_loans_df["tenure_years"].astype("Int64")

In [103]:
pd.set_option("display.max_rows", None)

silver_loans_df.dtypes.to_frame("Data tipi")

,Data tipi
loan_id,str
customer_id,str
application_date,datetime64[us]
product_type,str
amount,float64
term_months,int64
interest_rate,float64
status,str
rejection_reason,str
defaulted,Int64


indi bu cedveli sql e yazaq

In [104]:
silver_loans_df.to_sql(
    "loans", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_loans_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

17

Bronze dan transaction enriched faylini oxuyaq

In [105]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
   

In [106]:
silver_transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,3.5,0.0,14.8
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,7.2,0.3,10.8


In [107]:
silver_transactions_df.shape

(77396, 32)

In [108]:
silver_transactions_df.duplicated().sum()

np.int64(0)

In [109]:
silver_transactions_df.isna().sum()

transaction_id             0
customer_id                0
date                       0
time                       0
amount                     0
currency                   0
category                   0
channel                    0
age                        0
gender                     0
city                       0
branch                     0
tenure_years               0
credit_score               0
balance                    0
salary                     0
products_count             0
has_card                   0
is_active                  0
churned                    0
merge_date                 0
is_holiday                 0
holiday_name           68014
is_weekend                 0
is_workday                 0
is_pre_holiday             0
rate_currency          72100
rate_base              72100
rate                   72100
temperature_2m_mean        0
precipitation_sum          0
wind_speed_10m_max         0
dtype: int64

In [110]:
silver_transactions_df.groupby("currency")["rate"].apply(
    lambda x: x.isna().sum()
)

currency
AZN    72100
EUR        0
USD        0
Name: rate, dtype: int64

Bu normaldır. Çünki valyuta məzənnəsi cədvəlimizdə USD və EUR üçün məzənnələr var, AZN üçün isə yoxdur.

AZN-in AZN-ə məzənnəsi həmişə 1-dir. Ona görə bu boşluqları 1 ilə doldura bilərik.

In [111]:
mask = silver_transactions_df["currency"] == "AZN"
silver_transactions_df.loc[mask, "rate"] = 1

In [112]:
silver_transactions_df["rate"].isna().sum()

np.int64(0)

In [113]:
silver_transactions_df.groupby("is_holiday")["holiday_name"].apply(
    lambda x: x.isna().sum())

is_holiday
0    68014
1        0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyerler yoxdur

In [114]:
silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name     68014
rate_currency    72100
rate_base        72100
dtype: int64

In [115]:
mask = silver_transactions_df["currency"] == "AZN"

silver_transactions_df.loc[mask, "rate_currency"] = "AZN"
silver_transactions_df.loc[mask, "rate_base"] = "AZN"

silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name    68014
dtype: int64

In [116]:
silver_transactions_df.dtypes.to_frame("Data tipi")

,Data tipi
transaction_id,str
customer_id,str
date,str
time,str
amount,float64
currency,str
category,str
channel,str
age,int64
gender,str


In [118]:
silver_transactions_df["date"] = pd.to_datetime(
    silver_transactions_df["date"],
    format="%m/%d/%Y"
)

silver_transactions_df["merge_date"] = pd.to_datetime(
    silver_transactions_df["merge_date"]
)

In [119]:
silver_transactions_df[["date", "merge_date"]].dtypes

date          datetime64[us]
merge_date     datetime64[s]
dtype: object

In [122]:
silver_transactions_df.to_sql(
    "transactions", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_transactions_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

46

In [123]:
with engine.connect() as conn:
    silver_tables = pd.read_sql(
        text("""
            SELECT TABLE_NAME
            FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'silver'
            ORDER BY TABLE_NAME
        """),
        conn
    )

display(silver_tables)

,TABLE_NAME
0,loans
1,transactions
